In [1]:
import os
from tqdm import tqdm
import sys
if not hasattr(sys.modules[__name__], "cwd_changed"):
    os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(__name__))))
    sys.modules[__name__].cwd_changed = True

import warnings 
warnings.filterwarnings("ignore")
import logging
logging.getLogger('pgmpy').setLevel(logging.WARNING)



import pandas as pd
from utils.graph import dag_to_cpdag
import ast
# from metrics.graph import compare_dags
from utils.results import *
from utils.plotting import *

In [2]:
data_subsets = ["binary_data"] #"jpmf_data"
algorithm_subsets = ["hc", "ea_fg", "ea_og"] #, "ea_hc", "ges"]

In [3]:
import pickle
from pathlib import Path
from functools import lru_cache

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt

# ---------- widgets ----------
subset_dropdown = widgets.Dropdown(
    options=data_subsets,
    value=data_subsets[0],
    description="Subset:",
)

algo_dropdown = widgets.Dropdown(
    options=algorithm_subsets,
    value=algorithm_subsets[0],
    description="Algorithm:",
)

r_idx_slider = widgets.IntSlider(value=0, min=0, max=0, description="r_idx")
d_idx_slider = widgets.IntSlider(value=0, min=0, max=0, description="d_idx (filtered)")

filters_status = widgets.HTML("")
filters_box = widgets.VBox([])
reset_filters_btn = widgets.Button(description="Reset filters", icon="refresh")

out = widgets.Output()

# ---------- loading (cached so you don't reload on every tweak) ----------
BASE = Path("results")


@lru_cache(maxsize=None)
def load_results(subset: str, algo: str):
    p = BASE / algo / f"{subset}.pkl"
    with p.open("rb") as f:
        return pickle.load(f)


def _candidate_mapping_paths(subset: str, algo: str) -> list[Path]:
    return [
        Path("data") / "experiments"/ "RQ1_struct_jpmf" / subset / "grid_mapping.csv",
        Path("data") / "experiments"/ "RQ1_struct" / subset / "grid_mapping.csv",
        BASE / algo / "grid_mapping.csv",
        BASE / algo / subset / "grid_mapping.csv",
        BASE / "mappings" / subset / "grid_mapping.csv",
        Path("grid_mapping.csv"),
    ]


@lru_cache(maxsize=None)
def load_grid_mapping(subset: str, algo: str) -> pd.DataFrame | None:
    for p in _candidate_mapping_paths(subset, algo):
        if p.exists():
            df = pd.read_csv(p)
            if "i" in df.columns:
                df["i"] = pd.to_numeric(df["i"], errors="ignore")
            return df
    return None


def clamp_slider(slider: widgets.IntSlider, new_max: int):
    new_max = max(0, int(new_max))
    slider.max = new_max
    if slider.value > new_max:
        slider.value = new_max


def _get_dataset_i(metadata_by_id, d_idx: int) -> int:
    if isinstance(metadata_by_id, dict):
        for k in ("i", "dataset_id", "dataset", "id"):
            if k in metadata_by_id:
                try:
                    return int(metadata_by_id[k])
                except Exception:
                    return metadata_by_id[k]
    for attr in ("i", "dataset_id", "dataset", "id"):
        if hasattr(metadata_by_id, attr):
            try:
                return int(getattr(metadata_by_id, attr))
            except Exception:
                return getattr(metadata_by_id, attr)
    return int(d_idx)


def _mapping_row_for_i(mapping: pd.DataFrame | None, i_val: int):
    if mapping is None or len(mapping) == 0:
        return None, "grid_mapping: (not found)"

    df = mapping
    if "i" in df.columns:
        try:
            hit = df.loc[df["i"].astype(str) == str(i_val)]
        except Exception:
            hit = df.loc[df["i"] == i_val]
    else:
        try:
            hit = df.loc[df.index.astype(str) == str(i_val)]
        except Exception:
            hit = df.loc[df.index == i_val]

    if hit is None or len(hit) == 0:
        return None, f"grid_mapping: no row for i={i_val}"

    row = hit.iloc[0]
    parts = [f"{k}={v}" for k, v in row.to_dict().items() if str(k) != "i"]
    return row, " | ".join(parts) if parts else f"i={i_val}"


def _add_mapping_to_fig(fig, header: str, mapping_text: str):
    title = f"{header}\n{mapping_text}" if mapping_text else header
    fig.suptitle(title, fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.92])


# -------------------------------
# Dropdown-only filter widgets
# -------------------------------
_filter_specs: dict[str, dict] = {}  # col -> {"kind": ..., "widget": ...}

ALL = ("All", None)


def _is_numeric_series(s: pd.Series) -> bool:
    try:
        pd.to_numeric(s.dropna().head(50), errors="raise")
        return True
    except Exception:
        return False


def _make_dropdown_filter_for_col(df: pd.DataFrame, col: str):
    s = df[col]

    # numeric column -> dropdown of exact values if small, else dropdown of bins
    if len(s.dropna()) > 0 and _is_numeric_series(s):
        nums = pd.to_numeric(s, errors="coerce")
        uniq = np.sort(nums.dropna().unique())

        if len(uniq) == 0:
            w = widgets.HTML(f"<i>{col}: (no numeric values)</i>")
            return {"kind": "none", "widget": w}

        # small unique -> exact dropdown
        if len(uniq) <= 80:
            opts = [ALL] + [(str(float(v) if isinstance(v, np.floating) else v), float(v)) for v in uniq]
            w = widgets.Dropdown(options=opts, value=None, description=col, layout=widgets.Layout(width="420px"))
            return {"kind": "exact_numeric", "widget": w}

        # many unique -> quantile-bin dropdown (still dropdown, no sliders)
        q = np.linspace(0, 1, 21)  # 20 bins
        edges = np.unique(np.quantile(uniq, q))
        if len(edges) < 2:  # fallback
            opts = [ALL] + [(str(float(v)), float(v)) for v in uniq[:80]]
            w = widgets.Dropdown(options=opts, value=None, description=col, layout=widgets.Layout(width="420px"))
            return {"kind": "exact_numeric", "widget": w}

        bins = []
        for lo, hi in zip(edges[:-1], edges[1:]):
            if np.isclose(lo, hi):
                continue
            label = f"[{lo:.4g}, {hi:.4g}]"
            bins.append((label, (float(lo), float(hi))))

        opts = [ALL] + bins
        w = widgets.Dropdown(options=opts, value=None, description=col, layout=widgets.Layout(width="420px"))
        return {"kind": "bin_numeric", "widget": w}

    # categorical / object column -> dropdown if reasonable unique else combobox
    uniq = pd.Series(s.astype("string")).dropna().unique().tolist()
    uniq = [u for u in uniq if str(u) != "<NA>"]
    uniq = sorted(map(str, uniq))

    if len(uniq) == 0:
        w = widgets.HTML(f"<i>{col}: (no values)</i>")
        return {"kind": "none", "widget": w}

    if len(uniq) <= 200:
        opts = [ALL] + [(v, v) for v in uniq]
        w = widgets.Dropdown(options=opts, value=None, description=col, layout=widgets.Layout(width="420px"))
        return {"kind": "exact_str", "widget": w}

    # huge unique -> combobox (dropdown with typing). Typed values do substring match.
    w = widgets.Combobox(
        options=[""] + uniq[:2000],  # keep it snappy
        value="",
        description=col,
        placeholder="(All) type to filter",
        ensure_option=False,
        layout=widgets.Layout(width="520px"),
    )
    return {"kind": "combo_contains", "widget": w}


def rebuild_filters():
    global _filter_specs
    subset = subset_dropdown.value
    algo = algo_dropdown.value

    mapping = load_grid_mapping(subset, algo)
    _filter_specs = {}

    if mapping is None or len(mapping) == 0:
        filters_box.children = [widgets.HTML("<i>grid_mapping.csv not found (filters disabled)</i>")]
        return

    cols = [c for c in mapping.columns if str(c) != "i"]
    if not cols:
        filters_box.children = [widgets.HTML("<i>No filterable columns found (only 'i'?)</i>")]
        return

    cols = sorted(cols, key=lambda x: str(x))
    ws = []
    for col in cols:
        spec = _make_dropdown_filter_for_col(mapping, col)
        _filter_specs[col] = spec
        w = spec["widget"]

        if isinstance(w, (widgets.Dropdown, widgets.Combobox)):
            w.observe(render, names="value")

        ws.append(w)

    # simple layout: 2 columns if many filters
    if len(ws) <= 6:
        filters_box.children = ws
    else:
        left = widgets.VBox(ws[0::2])
        right = widgets.VBox(ws[1::2])
        filters_box.children = [widgets.HBox([left, right])]


def apply_filters(mapping: pd.DataFrame | None) -> pd.DataFrame | None:
    if mapping is None or len(mapping) == 0 or not _filter_specs:
        return mapping

    df = mapping.copy()
    mask = pd.Series(True, index=df.index)

    for col, spec in _filter_specs.items():
        kind = spec["kind"]
        w = spec["widget"]

        if kind == "none" or col not in df.columns:
            continue

        if kind == "exact_str":
            val = w.value
            if val is not None:
                mask &= (df[col].astype("string") == str(val))

        elif kind == "exact_numeric":
            val = w.value
            if val is not None:
                nums = pd.to_numeric(df[col], errors="coerce")
                mask &= np.isclose(nums, float(val), equal_nan=False)

        elif kind == "bin_numeric":
            rng = w.value
            if rng is not None:
                lo, hi = rng
                nums = pd.to_numeric(df[col], errors="coerce")
                mask &= nums.between(lo, hi)

        elif kind == "combo_contains":
            text = (w.value or "").strip()
            if text:
                mask &= df[col].astype("string").str.contains(text, case=False, na=False)

    return df.loc[mask]


def reset_filters(_=None):
    for col, spec in _filter_specs.items():
        w = spec["widget"]
        kind = spec["kind"]

        if kind in ("exact_str", "exact_numeric", "bin_numeric"):
            w.value = None
        elif kind == "combo_contains":
            w.value = ""

    render()


reset_filters_btn.on_click(reset_filters)


# ---------- render ----------
def sync_slider_ranges():
    res = load_results(subset_dropdown.value, algo_dropdown.value)
    clamp_slider(r_idx_slider, len(res) - 1)


def _filtered_dataset_indices(res, r: int, mapping: pd.DataFrame | None) -> tuple[list[int], int]:
    n_d = len(res[r].true_graphs)
    all_ds = list(range(n_d))

    if mapping is None or len(mapping) == 0:
        return all_ds, 0

    mf = apply_filters(mapping)
    mapping_rows = 0 if mf is None else len(mf)

    if mf is None or len(mf) == 0:
        # no matches in mapping -> fall back to all datasets (so you still see something)
        return all_ds, mapping_rows

    if "i" in mf.columns:
        allowed_i = set(mf["i"].astype(str).tolist())
    else:
        allowed_i = set(mf.index.astype(str).tolist())

    allowed_ds = []
    for d in all_ds:
        i_val = _get_dataset_i(res[r].metadata_by_id[d], d)
        if str(i_val) in allowed_i:
            allowed_ds.append(d)

    if len(allowed_ds) == 0:
        allowed_ds = all_ds

    return allowed_ds, mapping_rows


def render(*_):
    sync_slider_ranges()
    subset = subset_dropdown.value
    algo = algo_dropdown.value

    res = load_results(subset, algo)
    mapping = load_grid_mapping(subset, algo)

    r = r_idx_slider.value
    allowed_ds, mapping_rows = _filtered_dataset_indices(res, r, mapping)

    clamp_slider(d_idx_slider, len(allowed_ds) - 1)
    d_pos = d_idx_slider.value
    d = allowed_ds[d_pos]

    total_map = 0 if mapping is None else len(mapping)
    filters_status.value = (
        f"<b>Filters:</b> matched mapping rows: {mapping_rows}/{total_map} | "
        f"matching datasets in this replication: {len(allowed_ds)}/{len(res[r].true_graphs)}"
    )

    with out:
        out.clear_output(wait=True)

        true_graph = res[r].true_graphs[d]
        learned_graph = res[r].learned_graphs[d]
        metadata = res[r].metadata_by_id[d]

        i_val = _get_dataset_i(metadata, d)
        row, mapping_text = _mapping_row_for_i(mapping, i_val)

        fig = plot_three_side_by_side(
            figsize=(25, 10),
            dag=learned_graph,
            true_graph=true_graph,
            learned_graph=learned_graph,
            metadata=metadata,
            titles=("Learned DAG", "Learned vs True DAG", "Metadata Causal Network"),
        )

        header = (
            f"subset={subset} | algo={algo} | r_idx={r} | "
            f"d_idx={d} (filtered pos={d_pos}) | i={i_val}"
        )
        _add_mapping_to_fig(fig, header=header, mapping_text=mapping_text)

        plt.show()

        if row is not None:
            display(pd.DataFrame([row.to_dict()]))


def on_subset_algo_change(*_):
    rebuild_filters()
    d_idx_slider.value = 0
    r_idx_slider.value = 0
    render()


subset_dropdown.observe(on_subset_algo_change, names="value")
algo_dropdown.observe(on_subset_algo_change, names="value")
r_idx_slider.observe(render, names="value")
d_idx_slider.observe(render, names="value")

# Initial build + draw + layout
rebuild_filters()
render()

display(widgets.VBox([
    widgets.HBox([subset_dropdown, algo_dropdown]),
    widgets.HTML("<b>Grid-mapping filters (dropdowns)</b>"),
    filters_status,
    filters_box,
    reset_filters_btn,
    r_idx_slider,
    d_idx_slider,
    out
]))
